# Dimensionality Reduction

Coursework study extract, week 4.

Compare raw/scaled PCA, supervised LDA, mutual-information feature selection, and added noise. All learned transformations stay inside cross-validation pipelines. Final selection uses the training cross-validation table.

Adapted from completed code in `Module_4_Dimensionality_Reduction_Activity.ipynb`. Instructor-provided scaffolding and pair-work context are acknowledged; authorship of individual lines cannot be independently separated. See [ATTRIBUTION.md](../ATTRIBUTION.md). Teaching prompts, supplied solutions, saved outputs, and identifying metadata are omitted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42
print("Libraries ready.")


In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target

print("Shape:", X.shape)
print("Classes:", sorted(y.unique()))
X.head()


In [ ]:
display(X.describe().T)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Locked test:", X_test.shape)


In [ ]:
pca_unscaled = PCA(n_components=2)
X_train_pca = pca_unscaled.fit_transform(X_train)

print("Variance explained by each PC:",
    pca_unscaled.explained_variance_ratio_)

print(
    "Total variance retained:",
    f"{pca_unscaled.explained_variance_ratio_.sum():.2%}"
)

pc1_loadings = pd.Series(
    np.abs(pca_unscaled.components_[0]),
    index=X_train.columns,
    name="Absolute PC1 loading"
).sort_values(ascending=False)

display(pc1_loadings)

for label in sorted(y_train.unique()):
    mask = y_train == label
    plt.scatter(
        X_train_pca[mask, 0],
        X_train_pca[mask, 1],
        label=wine.target_names[label],
        alpha=0.75
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA without scaling — training data")
plt.legend(title="Cultivar")
plt.show()


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

print("Scaled train shape:", X_train_scaled.shape)

print("Standard deviations of the first five scaled features:")
display(pd.Series(
    X_train_scaled.std(axis=0)[:5],
    index=X_train.columns[:5],
    name="Scaled standard deviation"
))


In [ ]:
pca_full = PCA()
pca_full.fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
k_90 = np.argmax(cumulative_variance >= 0.90) + 1

print("Components needed for 90% variance:", k_90)

plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)
plt.axhline(0.90, color="red", linestyle="--", label="90% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA on standardized training data")
plt.legend()
plt.show()


In [ ]:
pca_scaled = PCA(n_components=2)
X_train_pca_scaled = pca_scaled.fit_transform(X_train_scaled)

print(
    "Variance retained by two scaled PCs:",
    f"{pca_scaled.explained_variance_ratio_.sum():.2%}"
)

for label in sorted(y_train.unique()):
    mask = y_train == label
    plt.scatter(
        X_train_pca_scaled[mask, 0],
        X_train_pca_scaled[mask, 1],
        label=wine.target_names[label],
        alpha=0.75
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA with standard scaling — training data")
plt.legend(title="Cultivar")
plt.show()


In [ ]:
lda = LinearDiscriminantAnalysis(n_components=2)

X_train_lda = lda.fit_transform(X_train_scaled, y_train)

for label in sorted(y_train.unique()):
    mask = y_train == label
    plt.scatter(
        X_train_lda[mask, 0],
        X_train_lda[mask, 1],
        label=wine.target_names[label],
        alpha=0.75
    )

plt.xlabel("LD1")
plt.ylabel("LD2")
plt.title("LDA — training data")
plt.legend(title="Cultivar")
plt.show()


In [ ]:
selector = SelectKBest(
    score_func=partial(mutual_info_classif, random_state=RANDOM_STATE),
    k=5
)

selector.fit(X_train, y_train)

selected_features = X_train.columns[selector.get_support()]

print("Five selected features:")
display(selected_features.to_list())


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

pipelines = {
    "Original 13 features": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000))
    ]),

    "Unscaled 2 PCs": Pipeline([
        ("pca", PCA(n_components=2)),
        ("model", LogisticRegression(max_iter=5000))
    ]),

    "Scaled PCA (90% variance)": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.90)),
        ("model", LogisticRegression(max_iter=5000))
    ]),

    "LDA (2 components)": Pipeline([
        ("scaler", StandardScaler()),
        ("lda", LinearDiscriminantAnalysis(n_components=2)),
        ("model", LogisticRegression(max_iter=5000))
    ]),

    "Top 5 MI features": Pipeline([
        ("selector", SelectKBest(
            score_func=partial(
                mutual_info_classif,
                random_state=RANDOM_STATE
            ),
            k=5
        )),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000))
    ])
}

rows = []

for name, pipeline in pipelines.items():
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    rows.append({
        "Representation": name,
        "Mean CV accuracy": scores.mean(),
        "Fold SD": scores.std()
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("Mean CV accuracy", ascending=False)
    .reset_index(drop=True)
)

display(cv_results.round(4))


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

noise = rng.normal(size=(X_train.shape[0], 50))
X_train_noise = np.hstack([X_train, noise])

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

rows = []

for name, features in {
    "Original 13 features": X_train,
    "13 features + 50 noise features": X_train_noise
}.items():
    scores = cross_val_score(
        knn_pipeline,
        features,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    rows.append({
        "Features": name,
        "Dimensions": features.shape[1],
        "Mean CV accuracy": scores.mean(),
        "Fold SD": scores.std()
    })

noise_results = pd.DataFrame(rows)
display(noise_results.round(4))


In [ ]:
selected_name = cv_results.iloc[0]["Representation"]
selected_model = pipelines[selected_name]

selected_model.fit(X_train, y_train)

locked_test_accuracy = selected_model.score(X_test, y_test)

print("Selected representation:", selected_name)
print(f"Locked-test accuracy: {locked_test_accuracy:.2%}")


## Review notes

Compare raw/scaled PCA, supervised LDA, mutual-information feature selection, and added noise. All learned transformations stay inside cross-validation pipelines. Final selection uses the training cross-validation table.

These are learning experiments, not production systems. Scores depend on the data split and package versions. Generated commentary in this publication is editorial context added during preparation, not a claim that it was part of the original class submission.